In [ ]:
import os
import zipfile
from huggingface_hub import hf_hub_download

# RunPod persistent storage directory
WORKSPACE_DIR = '/workspace/THUMOS14'
os.makedirs(WORKSPACE_DIR, exist_ok=True)

# Replace with the token you generated earlier (Read-only is fine here)
HF_TOKEN = "hf_YOUR_TOKEN_HERE" 
REPO_ID = "Sehrish05/THUMOS14"
ZIP_FILENAME = "THUMOS14.zip"

print(f"Downloading {ZIP_FILENAME} from Hugging Face...")
zip_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=ZIP_FILENAME,
    repo_type="dataset",
    local_dir="/workspace",
    token=HF_TOKEN
)

print("Extracting dataset... this will take a moment.")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(WORKSPACE_DIR)

print(f"Dataset successfully extracted to: {WORKSPACE_DIR}")

THUMOS14.zip:   0%|          | 0.00/978M [00:00<?, ?B/s]

Extracting dataset... this will take a moment.
Dataset successfully extracted to: /workspace/THUMOS14


In [ ]:
import os
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

class THUMOS14_I3D_Dataset(Dataset):
    def __init__(self, data_root, split='train', fps=25.0, clip_length=16):
        self.data_root = data_root
        self.split = split
        self.fps = fps
        self.clip_length = clip_length
        
        self.rgb_dir = os.path.join(data_root, 'features', split, 'rgb')
        self.flow_dir = os.path.join(data_root, 'features', split, 'flow')
        
        split_file = os.path.join(data_root, f'split_{split}.txt')
        with open(split_file, 'r') as f:
            self.video_ids = [line.strip() for line in f.readlines()]
            
        with open(os.path.join(data_root, 'gt.json'), 'r') as f:
            self.ground_truth = json.load(f)

        # --- THE RAM CACHE ---
        self.data_cache = []
        total_vids = len(self.video_ids)
        print(f"Loading {split} dataset into RAM ({total_vids} videos). This may take a minute...")
        
        for i, vid_id in enumerate(self.video_ids):
            # Print an update every 50 videos
            if i % 50 == 0 and i > 0:
                print(f"Progress: Loaded {i} / {total_vids} videos...")

            try:
                rgb_feat = np.load(os.path.join(self.rgb_dir, f'{vid_id}.npy'))
                flow_feat = np.load(os.path.join(self.flow_dir, f'{vid_id}.npy'))
                fused_features = torch.tensor(np.concatenate((rgb_feat, flow_feat), axis=-1), dtype=torch.float32)
            except FileNotFoundError:
                continue # Skip missing videos silently during caching
                
            num_vectors = fused_features.shape[0] 
            target_mask = torch.zeros(num_vectors, dtype=torch.float32)
            
            if vid_id in self.ground_truth['database']:
                annotations = self.ground_truth['database'][vid_id]['annotations']
                for ann in annotations:
                    start_sec = float(ann['segment'][0])
                    end_sec = float(ann['segment'][1])
                    
                    start_idx = int((start_sec * self.fps) / self.clip_length)
                    end_idx = int((end_sec * self.fps) / self.clip_length)
                    
                    start_idx = max(0, min(start_idx, num_vectors - 1))
                    end_idx = max(0, min(end_idx, num_vectors - 1))
                    
                    if start_idx <= end_idx:
                        target_mask[start_idx:end_idx + 1] = 1.0
            
            # Store the fully processed tensors in memory
            self.data_cache.append((fused_features, target_mask, vid_id))
            
        print(f"Finished caching {len(self.data_cache)} videos successfully!")

    def __len__(self):
        return len(self.data_cache)

    def __getitem__(self, idx):
        # Instantly return the pre-loaded tensors from RAM
        return self.data_cache[idx]

In [ ]:
import multiprocessing

# 1. Point to the new RunPod workspace
DATA_ROOT = '/workspace/THUMOS14' 

# 2. Scale the hardware utilization
# On a T4, you used batch_size=1 and workers=0. 
# On an L40s/A5000/4090, we can push this significantly higher.
cpu_cores = min(multiprocessing.cpu_count(), 8) # Cap at 8 to prevent I/O bottlenecks
BATCH_SIZE = 4 # You can likely increase this to 8 or 16 depending on the specific GPU

train_dataset = THUMOS14_I3D_Dataset(DATA_ROOT, split='train')

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,
    num_workers=cpu_cores, 
    pin_memory=True 
)

features, mask, vid_id = next(iter(train_loader))

print(f"Loaded Batch Size: {features.shape[0]}")
print(f"Using CPU Workers: {cpu_cores}")
print(f"Feature Tensor Shape: {features.shape}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        """
        alpha: Balances the importance of positive/negative examples.
               Since actions are rare, alpha gives them more weight.
        gamma: The focusing parameter. Higher values penalize the model 
               more for missing the rare action frames.
        """
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # inputs are raw logits before sigmoid
        BCE_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss) # Probability of the correct class
        
        # Calculate Focal Loss
        F_loss = self.alpha * (1 - pt)**self.gamma * BCE_loss

        if self.reduction == 'mean':
            return torch.mean(F_loss)
        elif self.reduction == 'sum':
            return torch.sum(F_loss)
        else:
            return F_loss

In [ ]:
import torch
import torch.nn as nn
from transformers import MambaConfig, MambaModel

class BiMambaGlobalScanner(nn.Module):
    def __init__(self, input_dim=2048, d_model=256, d_state=16, d_conv=4, expand=2, kernel_size=7):
        super(BiMambaGlobalScanner, self).__init__()
        
        # 1. Dimensionality Reduction
        self.input_proj = nn.Linear(input_dim, d_model)
        
        # 2. Temporal Smoothing Bottleneck (1D Depthwise Convolution)
        # We use padding=kernel_size//2 to ensure the sequence length doesn't change
        self.temporal_smooth = nn.Conv1d(
            in_channels=d_model, 
            out_channels=d_model, 
            kernel_size=kernel_size, 
            padding=kernel_size // 2,
            groups=d_model # Depthwise convolution keeps it extremely lightweight
        )
        self.norm = nn.LayerNorm(d_model)
        
        # 3. The Core Mamba Blocks (Bidirectional)
        config = MambaConfig(
            hidden_size=d_model,
            state_size=d_state,
            conv_kernel=d_conv,
            expand=expand
        )
        self.mamba_fwd = MambaModel(config)
        self.mamba_bwd = MambaModel(config)
        
        # 4. The Classification Head
        # Note: d_model is multiplied by 2 because we concatenate forward and backward states
        self.classifier = nn.Linear(d_model * 2, 1)

    def forward(self, x):
        # x shape: (Batch, Sequence_Length, 2048)
        
        # Project inputs
        x = self.input_proj(x)  # Shape: (Batch, Sequence_Length, 256)
        
        # Apply Temporal Smoothing
        # Conv1d expects shape (Batch, Channels, Sequence_Length), so we transpose
        x_t = x.transpose(1, 2)
        x_t = self.temporal_smooth(x_t)
        x_smoothed = x_t.transpose(1, 2)
        
        # Add a residual connection and normalize to stabilize training
        x = self.norm(x + x_smoothed)
        
        # Forward Temporal Scan
        out_fwd = self.mamba_fwd(inputs_embeds=x).last_hidden_state
        
        # Backward Temporal Scan
        # Flip the sequence along the time dimension (dim=1)
        x_reversed = torch.flip(x, dims=[1])
        out_bwd = self.mamba_bwd(inputs_embeds=x_reversed).last_hidden_state
        
        # Flip the backward output back to the original chronological order
        out_bwd = torch.flip(out_bwd, dims=[1])
        
        # Combine past and future context! 
        hidden_states = torch.cat([out_fwd, out_bwd], dim=-1) # Shape: (Batch, Sequence, 512)
        
        # Classify each timestep
        logits = self.classifier(hidden_states) # Shape: (Batch, Sequence, 1)
        
        # Remove the last dimension so it matches the target mask shape
        return logits.squeeze(-1)

In [ ]:
import torch
import torch.optim as optim
from tqdm.auto import tqdm

# Hyperparameters
EPOCHS = 10
LEARNING_RATE = 1e-4
MAX_SEQ_LEN = 1024 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print(f"🚀 Training on: {torch.cuda.get_device_name(DEVICE)} with {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM")

mamba_model = BiMambaGlobalScanner().to(DEVICE)
criterion = FocalLoss(alpha=0.85, gamma=2.0).to(DEVICE) 
optimizer = optim.AdamW(mamba_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

scaler = torch.amp.GradScaler()

def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss, total_recall, valid_batches = 0, 0, 0
    pbar = tqdm(dataloader, desc="Training Epoch")

    for features, targets, _ in pbar:
        seq_len = features.shape[1]
        if seq_len > MAX_SEQ_LEN:
            start_idx = torch.randint(0, seq_len - MAX_SEQ_LEN, (1,)).item()
            features = features[:, start_idx : start_idx + MAX_SEQ_LEN, :]
            targets = targets[:, start_idx : start_idx + MAX_SEQ_LEN]

        features, targets = features.to(device), targets.to(device)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):
            logits = model(features)
            loss = criterion(logits, targets)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        with torch.no_grad():
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float() 
            actual_positives = targets.sum().item()
            if actual_positives > 0:
                true_positives = (preds * targets).sum().item()
                recall = true_positives / actual_positives
                total_recall += recall
                valid_batches += 1
            else:
                recall = 0.0 

        pbar.set_postfix(Loss=f"{loss.item():.4f}", Recall=f"{recall:.4f}")

    return total_loss / len(dataloader), total_recall / valid_batches if valid_batches > 0 else 0

best_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    print(f"\n--- Epoch {epoch}/{EPOCHS} ---")
    avg_loss, avg_recall = train_epoch(mamba_model, train_loader, optimizer, criterion, DEVICE)
    print(f"Epoch {epoch} Complete | Avg Loss: {avg_loss:.4f} | Avg Recall: {avg_recall:.4f}")
    
    # --- NEW: SAVE PROGRESS AFTER EVERY EPOCH ---
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': mamba_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': avg_loss,
    }
    # Save the latest epoch (overwrites previous to save space)
    torch.save(checkpoint, "/workspace/mamba_scanner_latest.pth")
    
    # Save a separate copy if it's the best performing model so far
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(mamba_model.state_dict(), "/workspace/mamba_scanner_best.pth")
        print("🌟 New best model saved!")

print("\nStage 1 Training Complete!")

In [ ]:
import random

class TransformerRefinerDataset(Dataset):
    def __init__(self, data_root, split='train', window_size=64, fps=25.0, clip_length=16):
        self.window_size = window_size
        self.rgb_dir = os.path.join(data_root, 'features', split, 'rgb')
        self.flow_dir = os.path.join(data_root, 'features', split, 'flow')
        
        with open(os.path.join(data_root, 'gt.json'), 'r') as f:
            self.ground_truth = json.load(f)
            
        self.snippets = []
        
        # Build dataset by extracting windows around ground truth actions
        print(f"Building Refiner Dataset ({split})...")
        for vid_id, data in self.ground_truth['database'].items():
            if data['subset'] != split: continue
            
            try:
                rgb_feat = np.load(os.path.join(self.rgb_dir, f'{vid_id}.npy'))
                flow_feat = np.load(os.path.join(self.flow_dir, f'{vid_id}.npy'))
                features = torch.tensor(np.concatenate((rgb_feat, flow_feat), axis=-1), dtype=torch.float32)
            except FileNotFoundError:
                continue
                
            num_vecs = features.shape[0]
            
            for ann in data['annotations']:
                start_idx = int((float(ann['segment'][0]) * fps) / clip_length)
                end_idx = int((float(ann['segment'][1]) * fps) / clip_length)
                
                # Center a 64-frame window around the action, with some random jitter for robustness
                center = (start_idx + end_idx) // 2
                jitter = random.randint(-4, 4)
                w_start = center - (window_size // 2) + jitter
                w_end = w_start + window_size
                
                # Calculate normalized targets (0.0 to 1.0) inside this window
                t_start = max(0.0, min(1.0, (start_idx - w_start) / window_size))
                t_end = max(0.0, min(1.0, (end_idx - w_start) / window_size))
                
                snippet = torch.zeros((window_size, 2048), dtype=torch.float32)
                safe_start, safe_end = max(0, w_start), min(num_vecs, w_end)
                insert_start = safe_start - w_start
                insert_end = insert_start + (safe_end - safe_start)
                
                if safe_end > safe_start:
                    snippet[insert_start:insert_end] = features[safe_start:safe_end]
                    self.snippets.append((snippet, torch.tensor([t_start, t_end], dtype=torch.float32)))

    def __len__(self):
        return len(self.snippets)

    def __getitem__(self, idx):
        return self.snippets[idx]

# Hardware scaling: Large batch sizes for fast Transformer training
refiner_dataset = TransformerRefinerDataset(DATA_ROOT, split='train', window_size=64)
refiner_loader = DataLoader(
    refiner_dataset, 
    batch_size=128, # L40s/4090 can easily handle 128+ here
    shuffle=True, 
    num_workers=cpu_cores,
    pin_memory=True
)

snippets, targets = next(iter(refiner_loader))
print(f"Total Action Snippets: {len(refiner_dataset)}")
print(f"Batch Snippet Shape: {snippets.shape}")

In [ ]:
import torch
import torch.nn as nn
import math

class TransformerRefiner(nn.Module):
    def __init__(self, input_dim=2048, d_model=256, nhead=4, num_layers=1, window_size=64):
        super(TransformerRefiner, self).__init__()
        
        # 1. Dimensionality Reduction
        self.input_proj = nn.Linear(input_dim, d_model)
        
        # 2. NEW: Local Temporal Inductive Bias
        # This helps the Transformer see local frame-to-frame motion changes
        self.local_conv = nn.Conv1d(
            in_channels=d_model, 
            out_channels=d_model, 
            kernel_size=3, 
            padding=1, 
            groups=d_model
        )
        self.norm = nn.LayerNorm(d_model)
        
        # 3. Positional Encoding
        self.pos_encoder = nn.Parameter(torch.zeros(1, window_size, d_model))
        
        # 4. The Dense Attention Block
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=d_model * 4, 
            dropout=0.1, 
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 5. The Boundary Regressor Head
        self.regressor = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, 2),
            nn.Sigmoid() 
        )

    def forward(self, x):
        # x shape: (Batch, 64, 2048)
        x = self.input_proj(x) # Shape: (Batch, 64, 256)
        
        # Apply Local Convolution smoothing
        x_t = x.transpose(1, 2)
        x_t = self.local_conv(x_t)
        x_conv = x_t.transpose(1, 2)
        
        # Residual connection + Norm, then add Positional Encoding
        x = self.norm(x + x_conv) + self.pos_encoder
        
        # Apply dense self-attention
        x = self.transformer(x) # Shape: (Batch, 64, 256)
        
        # Temporal Pooling & Predict
        x_pooled = x.mean(dim=1) # Shape: (Batch, 256)
        boundaries = self.regressor(x_pooled) # Shape: (Batch, 2)
        
        return boundaries

In [ ]:
class GIoULoss1D(nn.Module):
    def __init__(self, eps=1e-6):
        super(GIoULoss1D, self).__init__()
        self.eps = eps

    def forward(self, preds, targets):
        # Ensure start is always strictly before end
        pred_start = torch.min(preds[:, 0], preds[:, 1])
        pred_end = torch.max(preds[:, 0], preds[:, 1])
        
        target_start = targets[:, 0]
        target_end = targets[:, 1]

        # Calculate Intersection
        intersect_start = torch.max(pred_start, target_start)
        intersect_end = torch.min(pred_end, target_end)
        intersect = torch.clamp(intersect_end - intersect_start, min=0)

        # Calculate Union
        pred_area = pred_end - pred_start
        target_area = target_end - target_start
        union = pred_area + target_area - intersect + self.eps

        # Calculate basic IoU
        iou = intersect / union

        # Calculate the smallest enclosing window
        enclose_start = torch.min(pred_start, target_start)
        enclose_end = torch.max(pred_end, target_end)
        enclose_area = torch.clamp(enclose_end - enclose_start, min=self.eps)

        # Calculate GIoU
        giou = iou - ((enclose_area - union) / enclose_area)

        # We want to minimize the loss, so 1 - GIoU
        loss = 1.0 - giou
        
        return loss.mean()

In [ ]:
import torch.optim as optim
from tqdm.auto import tqdm

EPOCHS = 15
LEARNING_RATE = 1e-4

print(f"Training Refiner on: {torch.cuda.get_device_name(DEVICE)}")

refiner_model = TransformerRefiner().to(DEVICE)
criterion = GIoULoss1D().to(DEVICE) 
optimizer = optim.AdamW(refiner_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scaler = torch.amp.GradScaler()

def train_refiner_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss, total_mae = 0, 0 
    pbar = tqdm(dataloader, desc="Training Refiner")

    for snippets, targets in pbar:
        snippets, targets = snippets.to(device), targets.to(device)
        optimizer.zero_grad(set_to_none=True)
        
        with torch.amp.autocast('cuda'):
            predictions = model(snippets)
            loss = criterion(predictions, targets)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        with torch.no_grad():
            mae = torch.abs(predictions - targets).mean().item()
            total_mae += mae

        pbar.set_postfix(Loss=f"{loss.item():.4f}", MAE=f"{mae:.4f}")

    return total_loss / len(dataloader), total_mae / len(dataloader)

best_mae = float('inf')

for epoch in range(1, EPOCHS + 1):
    avg_loss, avg_mae = train_refiner_epoch(refiner_model, refiner_loader, optimizer, criterion, DEVICE)
    print(f"Epoch {epoch} Complete | Avg Loss: {avg_loss:.4f} | Avg MAE: {avg_mae:.4f}")
    
    # --- NEW: SAVE PROGRESS AFTER EVERY EPOCH ---
    torch.save(refiner_model.state_dict(), "/workspace/transformer_refiner_latest.pth")
    
    if avg_mae < best_mae:
        best_mae = avg_mae
        torch.save(refiner_model.state_dict(), "/workspace/transformer_refiner_best.pth")
        print("🌟 New best refiner saved!")

print("\nStage 2 Training Complete!")

In [ ]:
# --- Configuration ---
SPLIT = 'test'
FPS = 25.0
CLIP_LENGTH = 16
WINDOW_SIZE = 64

print("Loading Models for Inference...")

# FIX: Changed from MambaGlobalScanner to BiMambaGlobalScanner
mamba_model = BiMambaGlobalScanner().to(DEVICE)
mamba_model.load_state_dict(torch.load("/workspace/mamba_scanner_stage1.pth", map_location=DEVICE, weights_only=True))
mamba_model.eval()

refiner_model = TransformerRefiner().to(DEVICE)
refiner_model.load_state_dict(torch.load("/workspace/transformer_refiner_stage2.pth", map_location=DEVICE, weights_only=True))
refiner_model.eval()

rgb_dir = os.path.join(DATA_ROOT, 'features', SPLIT, 'rgb')
flow_dir = os.path.join(DATA_ROOT, 'features', SPLIT, 'flow')

with open(os.path.join(DATA_ROOT, f'split_{SPLIT}.txt'), 'r') as f:
    test_videos = [line.strip() for line in f.readlines()]

final_predictions = {}

with torch.no_grad():
    for vid_id in tqdm(test_videos, desc="Evaluating Test Set"):
        try:
            rgb_feat = np.load(os.path.join(rgb_dir, f'{vid_id}.npy'))
            flow_feat = np.load(os.path.join(flow_dir, f'{vid_id}.npy'))
            video_features = torch.tensor(np.concatenate((rgb_feat, flow_feat), axis=-1), dtype=torch.float32).unsqueeze(0).to(DEVICE)
        except FileNotFoundError:
            continue
            
        num_vecs = video_features.shape[1]
        if num_vecs == 0: continue

        final_predictions[vid_id] = []

        # STAGE I: Global Scan
        mamba_logits = mamba_model(video_features)
        mamba_probs = torch.sigmoid(mamba_logits).squeeze(0).cpu().numpy()
        
        is_action = mamba_probs > 0.5
        padded = np.pad(is_action, (1, 1), 'constant')
        diffs = np.diff(padded.astype(int))
        starts = np.where(diffs == 1)[0]
        ends = np.where(diffs == -1)[0] - 1 

        # STAGE II: Local Refinement
        for s_idx, e_idx in zip(starts, ends):
            center_idx = (s_idx + e_idx) // 2
            w_start = center_idx - (WINDOW_SIZE // 2)
            w_end = w_start + WINDOW_SIZE
            
            snippet = torch.zeros((1, WINDOW_SIZE, 2048), dtype=torch.float32).to(DEVICE)
            
            safe_start, safe_end = max(0, w_start), min(num_vecs, w_end)
            insert_start, insert_end = safe_start - w_start, (safe_start - w_start) + (safe_end - safe_start)
            
            if safe_end > safe_start:
                snippet[0, insert_start:insert_end] = video_features[0, safe_start:safe_end]
                
            pred_bounds = refiner_model(snippet).squeeze(0).cpu().numpy()
            abs_start_idx = max(0, min(num_vecs, w_start + (pred_bounds[0] * WINDOW_SIZE)))
            abs_end_idx = max(0, min(num_vecs, w_start + (pred_bounds[1] * WINDOW_SIZE)))
            
            if abs_start_idx >= abs_end_idx: continue 
                
            start_sec = (abs_start_idx * CLIP_LENGTH) / FPS
            end_sec = (abs_end_idx * CLIP_LENGTH) / FPS
            confidence = float(np.max(mamba_probs[s_idx:e_idx+1]))
            
            final_predictions[vid_id].append({
                "segment": [float(start_sec), float(end_sec)],
                "score": confidence
            })

# Save to RunPod Workspace
with open("/workspace/submission_predictions.json", "w") as f:
    json.dump({"results": final_predictions}, f, indent=4)

print("\nInference Complete! Saved to /workspace/submission_predictions.json")

In [ ]:
import json
import numpy as np

GT_FILE = '/workspace/THUMOS14/gt.json' 
PRED_FILE = '/workspace/submission_predictions.json'
IOU_THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7]

def compute_1d_iou(segment1, segment2):
    intersection_start = max(segment1[0], segment2[0])
    intersection_end = min(segment1[1], segment2[1])
    intersection = max(0, intersection_end - intersection_start)
    
    union = (segment1[1] - segment1[0]) + (segment2[1] - segment2[0]) - intersection
    if union <= 0: return 0.0
    return intersection / union

with open(GT_FILE, 'r') as f:
    ground_truth = json.load(f)['database']
    
with open(PRED_FILE, 'r') as f:
    predictions = json.load(f)['results']

print("Calculating Average Precision (AP) at various IoU thresholds...\n")

ap_scores = []

for iou_thresh in IOU_THRESHOLDS:
    all_scores, all_matches, total_gt_actions = [], [], 0
    
    for vid_id, preds in predictions.items():
        if vid_id not in ground_truth: continue
            
        gt_annos = ground_truth[vid_id]['annotations']
        gt_segments = [[float(ann['segment'][0]), float(ann['segment'][1])] for ann in gt_annos]
        total_gt_actions += len(gt_segments)
        
        preds = sorted(preds, key=lambda x: x['score'], reverse=True)
        gt_matched = [False] * len(gt_segments)
        
        for p in preds:
            all_scores.append(p['score'])
            best_iou, best_gt_idx = 0, -1
            
            for i, gt_seg in enumerate(gt_segments):
                if gt_matched[i]: continue
                iou = compute_1d_iou(p['segment'], gt_seg)
                if iou > best_iou:
                    best_iou, best_gt_idx = iou, i
                    
            if best_iou >= iou_thresh:
                all_matches.append(1)
                gt_matched[best_gt_idx] = True
            else:
                all_matches.append(0)

    sorted_indices = np.argsort(all_scores)[::-1]
    all_matches = np.array(all_matches)[sorted_indices]
    
    cumulative_tp = np.cumsum(all_matches)
    cumulative_fp = np.cumsum(1 - all_matches)
    
    precision = cumulative_tp / (cumulative_tp + cumulative_fp + 1e-8)
    recall = cumulative_tp / (total_gt_actions + 1e-8)
    
    ap = sum(np.max(precision[recall >= t]) for t in np.arange(0, 1.1, 0.1) if np.sum(recall >= t) > 0) / 11.0
        
    ap_scores.append(ap)
    print(f"AP @ IoU {iou_thresh:.1f} : {ap * 100:.2f}%")

print("-" * 30)
print(f"Average mAP : {np.mean(ap_scores) * 100:.2f}%")